# ASRO pathology check - does harm appear on data from a NEW hospital?

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then **Runtime -> Run all**.
The first run downloads Camelyon17-WILDS (about 10 GB, roughly 15-30 minutes, only once per session); the analysis then takes about 15 minutes.

**Fresh start:** change settings only in **Cell 1**, then Runtime -> Restart session -> Run all.

**Question for the proposal:** the pilot study found that harmful directions are rare on normal images. ASRO expects harm under a REAL shift.
Camelyon17 has tissue patches (tumour / normal) from five hospitals. We train a simple classifier on patches from three hospitals
and test the same model on (a) new patches from those hospitals and (b) patches from a hospital it has never seen.

| Cells | What they do |
|---|---|
| 1-5 | Settings, download, the two models, the patches, shared functions |
| 6 | Concentration per layer: seen hospitals vs new hospital |
| 7 | Simple classifier (linear probe) trained on the seen hospitals; accuracy on seen vs new |
| 8 | Exact loss change when damping each top-8 direction (+ random directions for tau), seen vs new |
| 9 | ONE summary block - copy it and paste it into the Claude app |
| 10 | Zip all results |

Results go to the folder `asro_pathology_check`.

In [ ]:
# Cell 1 - SETUP AND SETTINGS (the only cell you ever need to edit)
# What this cell does and why: installs the packages (including WILDS, which provides Camelyon17), fixes the seed and keeps
# ALL settings in one place, so a fresh start never needs changes anywhere else.
!pip -q install wilds transformers torchvision scipy pandas
import os, time, shutil, random
import numpy as np, pandas as pd, torch, torch.nn.functional as F

SEED = 0
SEEN_HOSPITALS = [0, 3, 4]       # the WILDS training hospitals: the classifier is trained here
NEW_HOSPITAL = 2                 # the WILDS test hospital: never seen in training
N_TRAIN_PER_CLASS = 50           # training patches per class per seen hospital (2 x 50 x 3 = 300)
N_EVAL_PER_CLASS = 32            # evaluation patches per class in each group (64 seen + 64 new)
PILOT_LAYERS = [8, 9, 10, 12]
K = 8
DELTAS = [0.1, 1.0]
N_RANDOM = 8
OUTLIER_FACTOR = 10
MODEL_NAMES = {"standard": "facebook/dinov2-base", "registers": "facebook/dinov2-with-registers-base"}
FRESH_START = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = "/content" if os.path.isdir("/content") else "."
OUT = os.path.join(ROOT, "asro_pathology_check"); DATA_DIR = os.path.join(ROOT, "data")
if FRESH_START and os.path.isdir(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
T_START = time.time()
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - Runtime -> Change runtime type -> T4 GPU")
print("Expected: device: cuda | Tesla T4")

In [ ]:
# Cell 2 - DOWNLOAD Camelyon17-WILDS (about 10 GB; only the first time in a session)
# What this cell does and why: Camelyon17-WILDS (Koh et al., 2021; Bandi et al., 2019) has 96 x 96 tissue patches labelled
# tumour / normal, with the hospital each patch came from - exactly the hospital-to-hospital shift ASRO is meant for.
from wilds import get_dataset
try:
    ds = get_dataset(dataset="camelyon17", download=True, root_dir=DATA_DIR)
except Exception as e:
    raise RuntimeError("Camelyon17 download failed. Check that the disk is not full (folder icon), then "
                       "Runtime -> Restart session -> Run all. Details: " + str(e))
hosp = ds.metadata_array[:, ds.metadata_fields.index("hospital")].numpy()
y_all = ds.y_array.numpy()
print("patches:", len(ds), "| per hospital:", {int(h): int((hosp == h).sum()) for h in np.unique(hosp)})
print("Expected: about 455,000 patches from hospitals 0-4")

In [ ]:
# Cell 3 - LOAD THE TWO MODELS (frozen), the same as in the pilot study
# What this cell does and why: the same DINOv2-base models as the pilot study, so results can be compared directly.
from transformers import AutoImageProcessor, AutoModel
MODELS = {}
for name, hf in MODEL_NAMES.items():
    proc = AutoImageProcessor.from_pretrained(hf)
    mdl = AutoModel.from_pretrained(hf).to(device).eval()
    for p in mdl.parameters():
        p.requires_grad_(False)
    MODELS[name] = dict(model=mdl, processor=proc, n_front=1 + getattr(mdl.config, "num_register_tokens", 0))
    print(f"{name:9s}: {hf}, {len(mdl.encoder.layer)} layers")

In [ ]:
# Cell 4 - PICK THE PATCHES (fixed by the seed): training set, "seen hospitals" test set, "new hospital" test set
# What this cell does and why: balanced tumour / normal patches. The two test sets have the same size and balance,
# so any difference between them comes from the hospital, not from the sampling.
rng = np.random.default_rng(SEED)
def pick(hospitals, n_per_class, exclude=set()):
    idx = []
    for h in hospitals:
        for c in [0, 1]:
            pool = np.setdiff1d(np.where((hosp == h) & (y_all == c))[0], np.fromiter(exclude, int, len(exclude)))
            idx += list(rng.choice(pool, n_per_class, replace=False))
    return idx
train_idx = pick(SEEN_HOSPITALS, N_TRAIN_PER_CLASS)
seen_idx = pick(SEEN_HOSPITALS, max(1, N_EVAL_PER_CLASS // len(SEEN_HOSPITALS)) + 1, exclude=set(train_idx))
seen_idx = list(rng.permutation(seen_idx))
seen_idx = [i for i in seen_idx if y_all[i] == 0][:N_EVAL_PER_CLASS] + [i for i in seen_idx if y_all[i] == 1][:N_EVAL_PER_CLASS]
new_idx = pick([NEW_HOSPITAL], N_EVAL_PER_CLASS)
load = lambda ids: [ds[int(i)][0].convert("RGB") for i in ids]
DATA = {"train": (load(train_idx), torch.tensor(y_all[train_idx], device=device)),
        "seen": (load(seen_idx), torch.tensor(y_all[seen_idx], device=device)),
        "new": (load(new_idx), torch.tensor(y_all[new_idx], device=device))}
for k, (im, y) in DATA.items():
    print(f"{k:5s}: {len(im)} patches, tumour share {y.float().mean():.2f}")
print("Expected: train 300, seen 64, new 64; tumour share 0.50 each")

In [ ]:
# Cell 5 - SHARED FUNCTIONS (the same as in the pilot study)
# What this cell does and why: read the token matrix X of a layer, stable rank, the damping hook X(I - delta v v^T),
# and the classifier loss with or without damping.
def pixels(name, imgs):
    return MODELS[name]["processor"](images=imgs, return_tensors="pt")["pixel_values"].to(device)
def _get(o): return o[0] if isinstance(o, (tuple, list)) else o
def _put(o, new): return (new,) + tuple(o[1:]) if isinstance(o, (tuple, list)) else new
def hidden_states(name, pv, batch=16):
    model = MODELS[name]["model"]; out = {}
    for s in range(0, len(pv), batch):
        store, hs = {}, []
        for i, layer in enumerate(model.encoder.layer):
            hs.append(layer.register_forward_hook(lambda m, inp, o, i=i: store.__setitem__(i + 1, _get(o).detach().float().cpu())))
        try:
            with torch.no_grad(): model(pixel_values=pv[s:s + batch])
        finally:
            for h in hs: h.remove()
        for l, v in store.items(): out.setdefault(l, []).append(v)
    return {l: torch.cat(v) for l, v in sorted(out.items())}
def stable_rank(X):
    S = torch.linalg.svdvals(X); return (S ** 2).sum(-1) / S[..., 0] ** 2
def damping_hook(V, delta):
    state = {"start": 0}
    def hook(module, inputs, output):
        X = _get(output); b = X.shape[0]
        v = V[state["start"]:state["start"] + b].to(X.device, X.dtype); state["start"] += b
        return _put(output, X - delta * torch.einsum("btd,bd->bt", X, v).unsqueeze(-1) * v.unsqueeze(1))
    return hook
def features(name, pv, layer=None, V=None, delta=0.0, batch=16):
    model = MODELS[name]["model"]
    h = model.encoder.layer[layer - 1].register_forward_hook(damping_hook(V, delta)) if layer else None
    try:
        with torch.no_grad():
            return torch.cat([model(pixel_values=pv[s:s + batch]).pooler_output.float() for s in range(0, len(pv), batch)])
    finally:
        if h: h.remove()
def probe_loss(name, pv, y, layer=None, V=None, delta=0.0):
    logits = MODELS[name]["probe"](features(name, pv, layer, V, delta))
    return F.cross_entropy(logits, y, reduction="none").cpu(), (logits.argmax(-1) == y).cpu()
print("functions ready")

In [ ]:
# Cell 6 - CONCENTRATION PER LAYER: seen hospitals vs new hospital
# What this cell does and why: does the spectral concentration (stable rank, outlier tokens) look different on tissue
# from a hospital the model has not seen? Patch tokens only, as in the pilot study.
rows = []
for name, d in MODELS.items():
    d["pv"], d["hs"] = {}, {}
    for grp in ["seen", "new"]:
        d["pv"][grp] = pixels(name, DATA[grp][0]); d["hs"][grp] = hidden_states(name, d["pv"][grp])
        for l, H in d["hs"][grp].items():
            X = H[:, d["n_front"]:, :]; norms = X.norm(dim=-1); med = norms.median(1).values
            rows.append(dict(model=name, group=grp, layer=l, stable_rank=float(stable_rank(X).mean()),
                             stable_rank_centred=float(stable_rank(X - X.mean(1, keepdim=True)).mean()),
                             outliers_per_image=float((norms > OUTLIER_FACTOR * med[:, None]).sum(1).float().mean())))
P1 = pd.DataFrame(rows); P1.to_csv(f"{OUT}/P1_concentration_seen_vs_new.csv", index=False)
print(P1.pivot_table(index=["model", "layer"], columns="group", values="stable_rank").round(2).to_string())

In [ ]:
# Cell 7 - SIMPLE CLASSIFIER (linear probe, 768 -> 2: normal / tumour) trained on the SEEN hospitals only
# What this cell does and why: gives the loss we need, and shows how much accuracy drops on the new hospital
# (a real distribution shift, unlike blur or noise in the pilot study).
acc_rows = []
for name, d in MODELS.items():
    f_tr = features(name, pixels(name, DATA["train"][0])); y_tr = DATA["train"][1]
    mu, sd = f_tr.mean(0), f_tr.std(0) + 1e-6
    lin = torch.nn.Linear(f_tr.shape[1], 2).to(device)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-2, weight_decay=1e-4)
    for step in range(300):
        opt.zero_grad(); loss = F.cross_entropy(lin((f_tr - mu) / sd), y_tr); loss.backward(); opt.step()
    for p in lin.parameters(): p.requires_grad_(False)
    d["probe"] = lambda f, lin=lin, mu=mu, sd=sd: lin((f - mu) / sd)
    d["base"] = {}
    for grp in ["seen", "new"]:
        d["base"][grp] = probe_loss(name, d["pv"][grp], DATA[grp][1])
        acc_rows.append(dict(model=name, group=grp, accuracy=float(d["base"][grp][1].float().mean()),
                             mean_loss=float(d["base"][grp][0].mean())))
P2 = pd.DataFrame(acc_rows); P2.to_csv(f"{OUT}/P2_accuracy_seen_vs_new.csv", index=False)
print(P2.round(3).to_string(index=False))
print("Expected: accuracy on the new hospital LOWER than on the seen hospitals (that is the shift)")

In [ ]:
# Cell 8 - EXACT LOSS CHANGE when damping each top-8 direction (+ random directions = noise floor tau), seen vs new
# What this cell does and why: the key test. If harmful directions (damping LOWERS the loss) are more common on the new
# hospital than on the seen hospitals, harm appears where ASRO expects it.
rows = []; t0 = time.time()
for name, d in MODELS.items():
    for grp in ["seen", "new"]:
        pv, y = d["pv"][grp], DATA[grp][1]; base = d["base"][grp][0]
        for l in PILOT_LAYERS:
            _, S, Vh = torch.linalg.svd(d["hs"][grp][l], full_matrices=False)
            R = F.normalize(torch.randn(len(y), N_RANDOM, Vh.shape[-1], generator=torch.Generator().manual_seed(SEED + l)), dim=-1)
            for delta in DELTAS:
                top = torch.stack([probe_loss(name, pv, y, l, Vh[:, k, :], delta)[0] - base for k in range(K)], 1)
                rnd = torch.stack([probe_loss(name, pv, y, l, R[:, k, :], delta)[0] - base for k in range(N_RANDOM)], 1)
                tau = float(np.percentile(rnd.abs().numpy(), 95)); harm = top < -tau
                rows.append(dict(model=name, group=grp, layer=l, delta=delta, tau=tau,
                                 harmful=float(harm.float().mean()), useful=float((top > tau).float().mean()),
                                 patches_with_a_harmful_direction=float(harm.any(1).float().mean()),
                                 mean_gain_of_harmful=float(-top[harm].mean()) if harm.any() else 0.0))
        print(f"{name:9s} {grp:4s} done ({time.time() - t0:.0f}s)")
P3 = pd.DataFrame(rows); P3.to_csv(f"{OUT}/P3_harmful_seen_vs_new.csv", index=False)
print(P3.pivot_table(index=["model", "delta", "layer"], columns="group", values="harmful").round(3).to_string())
print("How to read it: 'new' clearly above 'seen' (and above ~0.05) -> harm appears under the hospital shift.")

In [ ]:
# Cell 9 - ONE SUMMARY: copy everything this cell prints and paste it into the Claude app
# What this cell does and why: the numbers the proposal needs, in one block (also saved as pathology_summary.md).
lines = [f"# ASRO pathology check - summary (Camelyon17-WILDS; seen hospitals {SEEN_HOSPITALS}, new hospital {NEW_HOSPITAL}; "
         f"{len(DATA['seen'][1])} + {len(DATA['new'][1])} test patches; seed {SEED})", "", "## Accuracy of the simple classifier"]
lines += [f"- {r.model:9s} {r.group:4s}: accuracy {r.accuracy:.3f}, mean loss {r.mean_loss:.3f}" for r in P2.itertuples()]
lines += ["", "## Concentration (stable rank, mean over patches)"]
for name in MODELS:
    for grp in ["seen", "new"]:
        q = P1[(P1.model == name) & (P1.group == grp)].set_index("layer")
        lines.append(f"- {name:9s} {grp:4s}: layers 9-11 {q.loc[[9, 10, 11], 'stable_rank'].min():.2f}-{q.loc[[9, 10, 11], 'stable_rank'].max():.2f}, "
                     f"final layer {q.stable_rank.iloc[-1]:.2f}, outliers/patch (max) {q.outliers_per_image.max():.2f}")
lines += ["", "## Directions that hurt the task (share of patch x direction cases; chance level about 0.05)"]
for (name, delta), g in P3.groupby(["model", "delta"]):
    s_, n_ = g[g.group == "seen"], g[g.group == "new"]
    lines.append(f"- {name:9s} delta {delta}: harmful seen {s_.harmful.mean():.3f} vs new {n_.harmful.mean():.3f}; "
                 f"patches with a harmful direction seen {s_.patches_with_a_harmful_direction.mean():.2f} vs new {n_.patches_with_a_harmful_direction.mean():.2f}")
lines.append(""); lines.append("## Per layer (delta 0.1): harmful seen / new")
for r in P3[P3.delta == 0.1].pivot_table(index=["model", "layer"], columns="group", values="harmful").reset_index().itertuples():
    lines.append(f"- {r.model:9s} layer {r.layer:2d}: {r.seen:.3f} / {r.new:.3f}")
lines += ["", f"(total run time {(time.time() - T_START) / 60:.0f} min)"]
open(f"{OUT}/pathology_summary.md", "w").write("\n".join(lines))
print("\n".join(lines)); print("\n# END - copy everything above and paste it into the Claude app")

In [ ]:
# Cell 10 - ZIP ALL RESULTS and download (run LAST)
# What this cell does and why: one zip with every CSV and the summary, ready for GitHub (results/pathology_check/).
zip_path = shutil.make_archive(OUT, "zip", OUT)
print("Files:"); [print("   ", f) for f in sorted(os.listdir(OUT))]
print("Zip file ready:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Not in Colab (or download blocked): folder icon on the left -> right-click the zip -> Download")